<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/01_learning/notebooks/sesion_01_ejercicio_mlp_multiclase_kaggle.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/01_learning/notebooks/sesion_01_ejercicio_mlp_multiclase_kaggle.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/jdmartinev-org/vision-model/studios/si7011-sesion01/code)

# SI7011 · Ejercicio: MLP multiclase con datos tabulares
**Juan David Martínez Vargas · Universidad EAFIT**

## Clasificación de variedades de fríjol · Dry Bean
A partir de medidas geométricas de un grano, predice su variedad. Usaremos el archivo tabular del [Dry Bean Dataset en Kaggle](https://www.kaggle.com/datasets/muratkokludataset/dry-bean-dataset), publicado por Murat Koklu. La fuente original es [UCI](https://archive.ics.uci.edu/dataset/602/dry+bean).

El archivo original contiene **13.611 registros, 16 variables numéricas y 7 clases**. Las variables fueron extraídas de imágenes; en este ejercicio recibimos esas medidas y entrenamos una **MLP**, sin procesar imágenes.

El ejercicio recorre las tres etapas de cualquier sistema de aprendizaje profundo, cada una con un **artefacto** verificable:

| Etapa | Artefacto |
|---|---|
| **Datos** | contrato de datos (columnas, tipos, rangos) y ficha del conjunto |
| **Entrenamiento** | estado del modelo seleccionado por validación |
| **Despliegue** | un paquete con modelo + preprocesamiento + etiquetas que predice sobre filas crudas |

### Trabajo del estudiante
Completa los siete `TODO`, ejecuta las verificaciones y responde las preguntas. Las funciones incompletas lanzan `NotImplementedError`: ese comportamiento es intencional. No continúes hasta completar cada tarea.

| Bloque | Minutos |
|---|---:|
| Carga e inspección | 5 |
| Partición y estandarización | 10 |
| Tensores y DataLoader | 10 |
| Arquitectura y pérdida | 10 |
| Entrenamiento | 15 |
| Evaluación e interpretación | 10 |
| Empaquetar y desplegar | 10 |

**Preparación previa (antes de la clase):** en **Colab**, guarda tu token de Kaggle como secreto `KAGGLE_API_TOKEN` (ver sección 1); en **Kaggle**, agrega el dataset con **Add Input**. CPU es suficiente.

## 1. Carga e inspección · 5 min

La celda de carga busca los datos en este orden y usa el primero que encuentre:

| Plataforma | De dónde salen los datos |
|---|---|
| **Kaggle** | el dataset agregado con **Add Input** (`/kaggle/input/...`), sin descargar |
| **Lightning Studio** | el archivo del Studio: `data/Dry_Bean_Dataset.xlsx` |
| **Local** | `Dry_Bean_Dataset.xlsx` o `data/Dry_Bean_Dataset.xlsx` junto al notebook |
| **Colab** | `kagglehub` con el dataset [`muratkokludataset/dry-bean-dataset`](https://www.kaggle.com/datasets/muratkokludataset/dry-bean-dataset) y tu token de Kaggle |
| sin archivo ni token | descarga desde UCI (CC BY 4.0) |

La celda imprime la **fuente** usada: revísala.

**En Colab, una sola vez:**
1. En [kaggle.com/settings](https://www.kaggle.com/settings) → *API* → **Generate New Token**; copia el token (empieza por `KGAT_`).
2. En Colab, ícono de llave 🔑 (*Secrets*) → **Add new secret** con nombre `KAGGLE_API_TOKEN` y el token como valor; activa *Notebook access*.

Nunca pegues el token en una celda: el notebook se comparte y el token quedaría expuesto.

In [ ]:
# Descomenta solo si falta algún paquete:
# %pip install torch pandas openpyxl scikit-learn matplotlib
from pathlib import Path
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device('cpu')
plt.rcParams.update({'figure.facecolor':'white', 'axes.spines.top':False,
                     'axes.spines.right':False, 'font.size':11})

KAGGLE_HANDLE = 'muratkokludataset/dry-bean-dataset'
UCI_URL = 'https://archive.ics.uci.edu/static/public/602/dry+bean+dataset.zip'


def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    import os
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):          # evita que credenciales viejas tengan prioridad
        os.environ.pop(k, None)
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()


def locate_data():
    candidates = [
        (Path('Dry_Bean_Dataset.xlsx'), 'archivo local'),
        (Path('data/Dry_Bean_Dataset.xlsx'), 'archivo local (data/)'),
        (Path('/teamspace/studios/this_studio/data/Dry_Bean_Dataset.xlsx'), 'Lightning Studio'),
        *[(p, 'input de Kaggle') for p in Path('/kaggle/input').glob('**/Dry_Bean_Dataset.xlsx')],
    ]
    for p, source in candidates:
        if p.exists():
            return p, source
    if kaggle_token() or Path('/kaggle/input').exists():
        try:
            import kagglehub                                    # preinstalado en Colab y Kaggle
            root = Path(kagglehub.dataset_download(KAGGLE_HANDLE))
            return next(root.rglob('Dry_Bean_Dataset.xlsx')), 'kagglehub'
        except Exception as e:
            print('kagglehub no disponible:', type(e).__name__, '-> se intenta UCI')
    import io, urllib.request, zipfile
    with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(UCI_URL, timeout=60).read())) as z:
        name = next(n for n in z.namelist() if n.endswith('Dry_Bean_Dataset.xlsx'))
        Path('Dry_Bean_Dataset.xlsx').write_bytes(z.read(name))
    return Path('Dry_Bean_Dataset.xlsx'), 'UCI'


DATA_PATH, SOURCE = locate_data()
df_raw = pd.read_excel(DATA_PATH)
print('Fuente:', SOURCE, '| archivo:', DATA_PATH.name, '| tamaño:', df_raw.shape)
print(df_raw.head().to_string(index=False))

In [ ]:
print('Faltantes:', int(df_raw.isna().sum().sum()))
print('Duplicados exactos:', int(df_raw.duplicated().sum()))
# Eliminar antes de dividir evita repartir copias exactas entre conjuntos.
df = df_raw.drop_duplicates().reset_index(drop=True)
print('Registros después de limpiar:', len(df))
print(df['Class'].value_counts().sort_index())
print(df.drop(columns='Class').describe().loc[['min','max']].T)
fig, ax = plt.subplots(figsize=(8,3))
df['Class'].value_counts().sort_index().plot.bar(ax=ax, color='#287dad')
ax.set(xlabel='Clase', ylabel='Registros', title='Distribución de clases')
fig.tight_layout(); plt.show()


### Contrato de datos

Antes de modelar fijamos qué entra al sistema. El **contrato** registra las columnas, su orden y su tipo, y lo usaremos dos veces: ahora, para revisar el archivo, y al final, para rechazar filas mal formadas **antes** de predecir. La **ficha** resume el conjunto y guarda una huella (*hash*) del archivo: si alguien cambia los datos, la huella cambia.

In [ ]:
import hashlib, json

FEATURES = [c for c in df.columns if c != 'Class']
CONTRACT = {'features': FEATURES, 'dtype': 'float32', 'target': 'Class',
            'classes': sorted(df['Class'].unique().tolist())}


def validate_rows(frame, contract=CONTRACT):
    # Devuelve las filas en el orden del contrato o falla con un mensaje claro.
    missing = [c for c in contract['features'] if c not in frame.columns]
    if missing:
        raise ValueError(f'Faltan columnas: {missing}')
    X = frame[contract['features']]
    if not all(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns):
        raise ValueError('Todas las variables deben ser numéricas.')
    if X.isna().any().any():
        raise ValueError(f'Hay valores faltantes en {X.columns[X.isna().any()].tolist()}')
    return X.to_numpy(dtype=np.float32, copy=True)


data_card = {'archivo': DATA_PATH.name,
             'sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()[:16],
             'registros': len(df), 'variables': len(FEATURES),
             'clases': df['Class'].value_counts().sort_index().to_dict()}
print(json.dumps(data_card, indent=1, ensure_ascii=False))


X_raw = validate_rows(df)
encoder = LabelEncoder()
y_all = encoder.fit_transform(df['Class']).astype(np.int64)
print('Codificación:', dict(zip(encoder.classes_, range(len(encoder.classes_)))))

**Respuesta:**

Las entradas son 16 medidas geométricas numéricas del grano y la salida es una de siete variedades (Class), codificada como un entero de 0 a 6. La clase menos frecuente es la que tenga menor conteo en la tabla anterior (normalmente Sira en este conjunto). Accuracy puede ocultar diferencias porque una clase mayoritaria domina el promedio y el modelo puede fallar proporcionalmente más en clases minoritarias.

## 2. TODO 1 · Partición y estandarización · 10 min
Implementa `prepare_data`:
1. Separar 60 % para entrenamiento y 40 % restante con `stratify=y` y `random_state=SEED`.
2. Dividir el restante en validación y prueba por partes iguales, también estratificando.
3. Ajustar `StandardScaler` **solo con entrenamiento** y transformar las tres particiones.
4. Devolver las entradas en `float32`, las etiquetas originales y el escalador, en el orden indicado.

No incorpores `Class` a las variables de entrada. Cada etiqueta es un entero entre 0 y 6; no necesitamos one-hot.

In [ ]:
def prepare_data(X, y):
    X_train, X_remaining, y_train, y_remaining = train_test_split(
        X, y, test_size=0.40, stratify=y, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(
        X_remaining, y_remaining, test_size=0.50, stratify=y_remaining,
        random_state=SEED)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)
    return X_train, X_val, X_test, y_train, y_val, y_test, scaler

X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_data(X_raw, y_all)
assert sum(map(len, [y_train, y_val, y_test])) == len(y_all)
assert X_train.shape[1] == 16 and X_train.dtype == np.float32
assert np.allclose(X_train.mean(axis=0), 0, atol=1e-4), 'Revisa dónde ajustaste el escalador.'
assert set(y_train) == set(y_val) == set(y_test) == set(range(7))
print('Particiones:', len(y_train), len(y_val), len(y_test))

**Respuesta:**

Ajustar con todo el dataset filtraría información de validación y prueba al entrenamiento, produciendo una evaluación demasiado optimista. Además, `Area` está en miles de unidades cuadradas mientras `Compactness` está cerca de 0–1; estandarizar evita que la escala grande domine la optimización.

## 3. TODO 2 · Tensores y DataLoader · 10 min
Implementa `make_loader` con `TensorDataset` y `DataLoader`:
- Entradas: `torch.float32`, forma `(N, 16)`.
- Etiquetas: `torch.long`, forma `(N,)`.
- Entrenamiento: lote de 128 y `shuffle=True`.
- Validación y prueba: lote de 512 y `shuffle=False`.
- Usa `num_workers=0` para esta práctica.

La función recibe `batch_size` y `shuffle`. Para repetir el orden de los lotes, pasa `generator=torch.Generator().manual_seed(SEED)` al loader.

In [ ]:
def make_loader(X, y, batch_size, shuffle):
    dataset = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                            torch.as_tensor(y, dtype=torch.long))
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, generator=generator)

train_loader = make_loader(X_train, y_train, batch_size=128, shuffle=True)
val_loader = make_loader(X_val, y_val, batch_size=512, shuffle=False)
test_loader = make_loader(X_test, y_test, batch_size=512, shuffle=False)
xb, yb = next(iter(train_loader))
assert xb.dtype == torch.float32 and yb.dtype == torch.long
assert xb.ndim == 2 and xb.shape[1] == 16 and yb.shape == (len(xb),)
print('Entradas:', xb.shape, '| objetivos:', yb.shape)

**Respuesta:**

DataLoader crea lotes, mezcla ejemplos y entrega tensores de forma uniforme. Mezclamos entrenamiento para evitar que el orden original influya en los lotes; validación solo mide y no necesita mezcla, por lo que un orden fijo facilita reproducibilidad.

## 4. TODO 3 y 4 · MLP y pérdida multiclase · 10 min
Construye una red **16 → 32 → 16 → 7** con ReLU después de cada capa oculta. La última capa devuelve siete logits, sin sigmoid ni softmax.

Para un lote de tamaño $B$:
- Logits: $(B,7)$, `float32`.
- Etiquetas: $(B,)$, `long`.
- Pérdida: un escalar con `CrossEntropyLoss`.

Las probabilidades se calculan después con softmax sobre la dimensión de clases. Para predecir una clase, el índice del mayor logit coincide con el de mayor probabilidad.

In [ ]:
class MulticlassMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 7)
        )
    def forward(self, x):
        return self.layers(x)

torch.manual_seed(SEED)
model = MulticlassMLP().to(device)
logits = model(xb.to(device))
assert logits.shape == (len(xb), 7), 'La salida debe tener un logit por clase.'
assert not any(isinstance(m, (nn.Softmax, nn.Sigmoid)) for m in model.modules())
print(model)
print('Parámetros:', sum(p.numel() for p in model.parameters()))

In [ ]:
def configure_training(model):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
    return criterion, optimizer

criterion, optimizer = configure_training(model)
initial_loss = criterion(logits, yb.to(device))
assert initial_loss.ndim == 0 and torch.isfinite(initial_loss)
print('Pérdida inicial:', initial_loss.item())

**Respuesta:**

Es una clasificación multiclase exclusiva: cada grano pertenece a una sola de siete clases. Por eso usamos siete logits y CrossEntropyLoss; un logit con BCEWithLogitsLoss corresponde a una salida binaria o multilabel. La primera capa tiene 16*32+32 = 544 parámetros y la última 16*7+7 = 119.

## 5. TODO 5 · Entrenamiento · 15 min
Completa `train_one_epoch`. Recorre los lotes, mueve entradas y etiquetas a `device`, limpia gradientes, calcula logits y pérdida, aplica backward y actualiza parámetros. Devuelve la pérdida media ponderada por el número de ejemplos: acumula `loss.item() * len(targets)`.

La evaluación y el control de épocas ya están preparados. Se conservará el estado con menor pérdida de validación. No se consulta prueba durante entrenamiento.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, total = 0.0, 0
    for features, targets in loader:
        features, targets = features.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(targets)
        total += len(targets)
    return loss_sum / total

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum, total = 0., 0
    targets_all, predictions_all = [], []
    for features, targets in loader:
        features, targets = features.to(device), targets.to(device)
        logits = model(features)
        loss_sum += criterion(logits, targets).item()*len(targets)
        total += len(targets)
        targets_all.extend(targets.cpu().tolist())
        predictions_all.extend(logits.argmax(dim=1).cpu().tolist())
    return {'loss':loss_sum/total,
            'accuracy':accuracy_score(targets_all,predictions_all),
            'macro_f1':f1_score(targets_all,predictions_all,average='macro',zero_division=0)}

EPOCHS = 30
history = {'train_loss':[], 'val_loss':[], 'val_accuracy':[], 'val_macro_f1':[]}
best_val_loss, best_state, best_epoch = float('inf'), None, None
for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    validation = evaluate(model, val_loader)
    history['train_loss'].append(train_loss)
    for key in ['loss','accuracy','macro_f1']:
        history['val_'+key].append(validation[key])
    if validation['loss'] < best_val_loss:
        best_val_loss = validation['loss']
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch+1
    if (epoch+1) % 10 == 0:
        print(f'Época {epoch+1}: train={train_loss:.3f}, '
              f'val={validation["loss"]:.3f}, F1 macro={validation["macro_f1"]:.3f}')
model.load_state_dict(best_state)
print('Época seleccionada por validación:', best_epoch)
print('Validación del estado seleccionado:', evaluate(model, val_loader))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,3.5))
axes[0].plot(history['train_loss'],label='Entrenamiento',color='#287dad')
axes[0].plot(history['val_loss'],label='Validación',color='#d58043')
axes[0].set(xlabel='Época (índice desde 0)',ylabel='Cross entropy');axes[0].legend()
axes[1].plot(history['val_accuracy'],label='Accuracy',color='#287dad')
axes[1].plot(history['val_macro_f1'],label='F1 macro',color='#167d78')
axes[1].set(xlabel='Época (índice desde 0)',ylabel='Métrica',ylim=(0,1));axes[1].legend()
fig.tight_layout();plt.show()

**Respuesta:**

`loss.backward()` calcula los gradientes y `optimizer.step()` actualiza los parámetros; `optimizer.zero_grad()` limpia los anteriores. La época seleccionada es `best_epoch`, la de menor pérdida de validación, que puede no ser la última. La prueba se reserva para una evaluación independiente y no para seleccionar el modelo.

## 6. TODO 6 · Evaluación final · 10 min
Implementa `predict_classes`: modo evaluación, sin registrar gradientes, recorrer lotes, obtener el índice del mayor logit por fila y concatenar predicciones como un arreglo de NumPy.

La matriz de confusión debe tener **filas reales y columnas predichas**. Se reportan accuracy, F1 macro y métricas por clase. Usa las etiquetas del `LabelEncoder` para interpretar los índices.

In [ ]:
@torch.no_grad()
def predict_classes(model, loader):
    model.eval()
    predictions = []
    for features, _ in loader:
        logits = model(features.to(device))
        predictions.append(logits.argmax(dim=1).cpu().numpy())
    return np.concatenate(predictions).astype(np.int64)

pred_test = predict_classes(model,test_loader)
assert pred_test.shape == y_test.shape and np.issubdtype(pred_test.dtype,np.integer)
assert set(pred_test).issubset(set(range(7)))
print('Accuracy prueba:', accuracy_score(y_test,pred_test))
print('F1 macro prueba:', f1_score(y_test,pred_test,average='macro',zero_division=0))
print(classification_report(y_test,pred_test,labels=np.arange(7),
                            target_names=encoder.classes_,zero_division=0))
fig,ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay.from_predictions(y_test,pred_test,labels=np.arange(7),
    display_labels=encoder.classes_,cmap='Blues',xticks_rotation=45,ax=ax,colorbar=False)
ax.set(xlabel='Clase predicha',ylabel='Clase real',title='Prueba · matriz de confusión')
fig.tight_layout();plt.show()

## 7. TODO 7 · Empaquetar y desplegar · 10 min

Un modelo entrenado **no es** un sistema que predice. Para usarlo con un registro nuevo necesitamos, juntos:

1. la arquitectura y los pesos;
2. el **preprocesamiento ajustado con entrenamiento** (media y desviación del escalador);
3. el orden de las columnas y el **mapa de índices a nombres de clase**.

Si falta cualquiera de los tres, el modelo recibe números con otra escala u otro orden y predice basura sin dar error. Guardamos todo en un solo archivo (`bundle`) con `torch.save`, usando solo tensores, listas y números.

**TODO 7.** Implementa `predict_raw(bundle, frame)`:
- validar `frame` con el contrato (usa `validate_rows` con `bundle['contract']`);
- estandarizar con `bundle['mean']` y `bundle['scale']`: $(x-\mu)/s$;
- reconstruir el modelo con `MulticlassMLP()` y `bundle['state_dict']`, en modo evaluación y sin gradientes;
- devolver un `DataFrame` con la columna `Class` (nombre de la clase) y `prob` (probabilidad de esa clase, con softmax).

In [ ]:
bundle = {'state_dict': model.state_dict(),
          'mean': torch.tensor(scaler.mean_, dtype=torch.float32),
          'scale': torch.tensor(scaler.scale_, dtype=torch.float32),
          'contract': CONTRACT, 'classes': encoder.classes_.tolist(),
          'data_card': data_card, 'best_epoch': best_epoch}
torch.save(bundle, 'drybean_mlp_bundle.pt')
print('Guardado drybean_mlp_bundle.pt:', round(Path('drybean_mlp_bundle.pt').stat().st_size / 1024, 1), 'KB')

In [ ]:
@torch.no_grad()
def predict_raw(bundle, frame):
    X = validate_rows(frame, bundle['contract'])
    features = (torch.from_numpy(X).to(device) - bundle['mean'].to(device)) / bundle['scale'].to(device)
    model = MulticlassMLP().to(device)
    model.load_state_dict(bundle['state_dict'])
    model.eval()
    probabilities = torch.softmax(model(features), dim=1)
    indices = probabilities.argmax(dim=1).cpu().numpy()
    return pd.DataFrame({'Class': [bundle['classes'][i] for i in indices],
                         'prob': probabilities.max(dim=1).values.cpu().numpy()},
                        index=frame.index)

### Prueba del paquete en un proceso "limpio"

Simulamos otro programa: cargamos solo el archivo y predecimos sobre filas **crudas** (sin estandarizar), tal como llegarían de producción. La prueba de consistencia exige que el paquete reproduzca exactamente las predicciones de prueba de la sección 6.

In [ ]:
loaded = torch.load('drybean_mlp_bundle.pt', weights_only=True)

# Filas crudas de prueba: deshacemos la estandarización para recuperar las medidas originales.
raw_test = pd.DataFrame(scaler.inverse_transform(X_test), columns=FEATURES)
out = predict_raw(loaded, raw_test.head(5))
print(out)

full = predict_raw(loaded, raw_test)
assert list(full.columns) == ['Class', 'prob'] and len(full) == len(raw_test)
assert (full['Class'].to_numpy() == encoder.inverse_transform(pred_test)).all(), 'El paquete no reproduce las predicciones.'
print('Consistencia: el paquete reproduce las', len(full), 'predicciones de prueba.')

# Un contrato que rechaza entradas mal formadas, antes de llegar al modelo:
try:
    predict_raw(loaded, raw_test.drop(columns=FEATURES[0]).head(2))
except ValueError as e:
    print('Rechazada como se esperaba ->', e)

**Respuesta:**

Con solo `state_dict` faltan el orden de columnas, la media y escala del `StandardScaler` y el mapa de índices a nombres. Una fila cruda todavía tiene dimensiones y tipos válidos, así que el modelo produce una salida sin error, pero puede predecir distinto porque recibe valores en una escala incompatible.

### Interpretación y entrega

1. El par más confundido es el de mayor valor fuera de la diagonal en la matriz de confusión; la celda se lee como fila real y columna predicha.
2. La clase con menor recall es la que tenga el menor valor en la columna `recall` del reporte. Compárala con su frecuencia: las clases menos frecuentes suelen tener menos ejemplos para aprender, aunque la relación debe justificarse con los valores observados.
3. F1 macro promedia el F1 de todas las clases con el mismo peso, mientras accuracy pondera implícitamente más a las clases frecuentes.
4. Del entrenamiento se reportan la época seleccionada y la pérdida de validación; de prueba, accuracy, F1 macro y la matriz de confusión. La prueba se interpreta después de seleccionar el estado y no se usa para ajustar decisiones.

## Referencias y alcance
- Koklu y Ozkan (2020), *Multiclass classification of dry beans using computer vision and machine learning techniques*, Computers and Electronics in Agriculture, 174, 105507. DOI: https://doi.org/10.1016/j.compag.2020.105507.
- [Kaggle · Dry Bean Dataset](https://www.kaggle.com/datasets/muratkokludataset/dry-bean-dataset).
- [UCI · Dry Bean](https://archive.ics.uci.edu/dataset/602/dry+bean), licencia CC BY 4.0.
- [PyTorch · CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).
- [PyTorch · Dataset y DataLoader](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html).

La partición es aleatoria y estratificada por registros. Este ejercicio no demuestra generalización a nuevas cámaras o lotes agrícolas; los identificadores de esos grupos no forman parte de esta tabla. Un único entrenamiento tampoco estima variabilidad entre semillas.